# Notebook 05: Conditional Image Generation — SD 1.5 + ConditionEncoder MLP

**Approach:** Replace CLIP text encoder in frozen Stable Diffusion 1.5 with a small MLP that maps glaze properties (surface, transparency, RGB, color_family) directly into cross-attention space (`77 × 768`). Train the MLP only using standard diffusion noise-prediction loss on the GlazyBench `image_generation` split.

**Architecture:**

- `ConditionEncoder` MLP: 25-dim condition → `Linear(256)` → `GELU` → `Linear(512)` → `GELU` → `Linear(768)` + learnable positional embeddings (`77 × 768`)
- Frozen: SD 1.5 UNet (860M), VAE, scheduling
- Trainable: ConditionEncoder only (~2M params, 2 MB on disk)

**Training:** 100 epochs, batch=16, 512×512 resolution, epsilon prediction
**GPU:** NVIDIA T4 (16GB, fp16 UNet/VAE, gradient checkpointing)
**Inference target:** CPU via OpenVINO INT8 (~2M param MLP → ~15s UNet inference)

**Outputs:** `condition_encoder.pt` + `condition_encoder.onnx` → copy to `backend/app/models/`


In [1]:
import os, sys, json, zipfile, shutil, subprocess, time, math, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

missing = []
for mod_name, import_name in [
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("huggingface_hub", "huggingface_hub"),
    ("accelerate", "accelerate"),
    ("lpips", "lpips"),
    ("torchmetrics", "torchmetrics"),
]:
    try:
        __import__(import_name)
    except ImportError:
        missing.append(mod_name)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import huggingface_hub
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler, DDIMScheduler
from huggingface_hub import hf_hub_download, login
import lpips
from torchmetrics.image.fid import FrechetInceptionDistance

print("All imports OK")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")
if n_gpu > 1:
    print(f"Using GPU 0 for training (2nd GPU idle — UNet is the bottleneck)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.1 MB/s eta 0:00:00


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


All imports OK
PyTorch 2.10.0+cu128, CUDA available: True
GPU 0: Tesla T4 | VRAM: 15.6 GB
GPU 1: Tesla T4 | VRAM: 15.6 GB
Using GPU 0 for training (2nd GPU idle — UNet is the bottleneck)


In [2]:
login()

In [3]:
REPO_ID = "AlpachinoNLP/GlazyBench"
DATA_DIR = Path("data/image_generation")
DATA_DIR.mkdir(exist_ok=True, parents=True)

for split in ["train", "test"]:
    split_dir = DATA_DIR / split
    split_dir.mkdir(exist_ok=True)
    for fname in ["images.zip", "targets.json", "metadata.json"]:
        dest = split_dir / fname
        if dest.exists() and dest.stat().st_size > 1000:
            print(f"  = {split}/{fname} already exists")
            continue
        cached = hf_hub_download(
            repo_id=REPO_ID,
            filename=f"image_generation/{split}/{fname}",
            repo_type="dataset",
        )
        shutil.copy2(cached, dest)
        print(f"  \u2713 {split}/{fname}")

    img_extract = split_dir / "images"
    if not img_extract.exists():
        zip_path = split_dir / "images.zip"
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(img_extract)
        jpg_count = len(list(img_extract.rglob("*.jpg")))
        print(f"  \u2713 {split}/images/ extracted ({jpg_count} files)")
    else:
        jpg_count = len(list(img_extract.rglob("*.jpg")))
        print(f"  = {split}/images/ already extracted ({jpg_count} files)")

print("Dataset download + extraction complete")

image_generation/train/images.zip:   0%|          | 0.00/481M [00:00<?, ?B/s]

  ✓ train/images.zip


targets.json: 0.00B [00:00, ?B/s]

  ✓ train/targets.json


metadata.json: 0.00B [00:00, ?B/s]

...

  ✓ test/images.zip


targets.json: 0.00B [00:00, ?B/s]

  ✓ test/targets.json


metadata.json: 0.00B [00:00, ?B/s]

  ✓ test/metadata.json
  ✓ test/images/ extracted (884 files)
Dataset download + extraction complete


In [6]:
def jload(path):
    with open(path) as f:
        return json.load(f)

train_targets = jload(DATA_DIR / "train" / "targets.json")
train_metadata = jload(DATA_DIR / "train" / "metadata.json")
test_targets = jload(DATA_DIR / "test" / "targets.json")
test_metadata = jload(DATA_DIR / "test" / "metadata.json")

print(f"Train targets: {len(train_targets)}  Train metadata: {len(train_metadata)}")
print(f"Test targets:  {len(test_targets)}  Test metadata:  {len(test_metadata)}")

train_meta_by_id = train_metadata  # already a dict keyed by id
test_meta_by_id = test_metadata


def filter_valid(targets, meta_by_id, split_name):
    valid = []
    for t in targets:
        sid = t["id"]
        if sid not in meta_by_id:
            continue
        if t.get("surface") is None or t.get("transparency") is None:
            continue
        if t.get("color_family") is None or t.get("color_rgb") is None:
            continue
        valid.append(t)
    print(f"{split_name}: {len(valid)} / {len(targets)} samples have all 4 properties")
    return valid

train_valid = filter_valid(train_targets, train_meta_by_id, "Train")
test_valid = filter_valid(test_targets, test_meta_by_id, "Test")

Train targets: 4490  Train metadata: 4490
Test targets:  443  Test metadata:  443
Train: 2826 / 4490 samples have all 4 properties
Test: 331 / 443 samples have all 4 properties


In [7]:
SURFACE_CLASSES = ["Dry Matte", "Glossy", "Matte", "Satin", "Satin-matte",
                  "Semi-glossy", "Semi-matte", "Smooth Matte", "Stony Matte"]
TRANSPARENCY_CLASSES = ["Opaque", "Semi-opaque", "Translucent", "Transparent"]
COLOR_FAMILY_CLASSES = ["Black", "Blue", "Gray", "Green", "Orange",
                       "Purple", "Red", "White", "Yellow"]

def build_condition_vector(t):
    surface_oh = [1.0 if c == t["surface"] else 0.0 for c in SURFACE_CLASSES]        # 9
    transp_oh = [1.0 if c == t["transparency"] else 0.0 for c in TRANSPARENCY_CLASSES]  # 4
    rgb = [v / 255.0 for v in t["color_rgb"][:3]]                                     # 3
    cf_oh = [1.0 if c == t["color_family"] else 0.0 for c in COLOR_FAMILY_CLASSES]    # 9
    return torch.tensor(surface_oh + transp_oh + rgb + cf_oh, dtype=torch.float32)

print(f"Condition vector dim: {build_condition_vector(train_valid[0]).shape[0]} (expect 25)")
print(f"Sample: {build_condition_vector(train_valid[0]).tolist()}")

Condition vector dim: 25 (expect 25)
Sample: [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.7450980544090271, 0.6392157077789307, 0.3803921639919281, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]


In [11]:
class GlazeDataset(Dataset):
    def __init__(self, targets, meta_by_id, img_dir, size=512, augment=False):
        self.targets = targets
        self.meta_by_id = meta_by_id
        self.img_dir = Path(img_dir)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        t = self.targets[idx]
        sid = t["id"]
        meta = self.meta_by_id[sid]
        rel = meta["image_filename"]
        path = self.img_dir / rel

        img = Image.open(path).convert("RGB").resize((self.size, self.size), Image.BILINEAR)

        if self.augment:
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)

        arr = np.array(img, dtype=np.float32) / 127.5 - 1.0
        img_t = torch.from_numpy(arr).permute(2, 0, 1)
        cond = build_condition_vector(t)
        return img_t, cond, sid

train_ds = GlazeDataset(train_valid, train_meta_by_id,
                        DATA_DIR / "train" / "images", augment=True)
test_ds = GlazeDataset(test_valid, test_meta_by_id,
                       DATA_DIR / "test" / "images", augment=False)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=False)

print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
print(f"Test:  {len(test_ds)} samples, {len(test_loader)} batches")

Train: 2826 samples, 177 batches
Test:  331 samples, 21 batches


In [12]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
device = torch.device("cuda")
DTYPE = torch.float16  # UNet + VAE in fp16 for T4 memory

print("Loading VAE (fp16)...")
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=DTYPE).to(device)
vae.requires_grad_(False)
vae.eval()

print("Loading UNet (fp16, gradient checkpointing)...")
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=DTYPE).to(device)
unet.requires_grad_(False)
unet.eval()
unet.enable_gradient_checkpointing()

print("Loading schedulers...")
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
infer_scheduler = DDIMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

vae_params = sum(p.numel() for p in vae.parameters()) / 1e6
unet_params = sum(p.numel() for p in unet.parameters()) / 1e6
print(f"VAE:  {vae_params:.1f}M params (frozen) — {vae.dtype}")
print(f"UNet: {unet_params:.1f}M params (frozen) — {unet.dtype}")
if torch.cuda.is_available():
    tot = torch.cuda.get_device_properties(0).total_memory / 1e9
    used = torch.cuda.memory_allocated(0) / 1e9
    print(f"VRAM: {used:.1f} GB / {tot:.1f} GB used after loading")

Loading VAE (fp16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading UNet (fp16, gradient checkpointing)...


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading schedulers...


scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

VAE:  83.7M params (frozen) — torch.float16
UNet: 859.5M params (frozen) — torch.float16
VRAM: 1.9 GB / 15.6 GB used after loading


In [15]:
class ConditionEncoder(nn.Module):
    def __init__(self, input_dim=25, hidden_dim=768, seq_len=77):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.GELU(),
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Linear(512, hidden_dim),
        )
        self.pos_embeds = nn.Parameter(torch.randn(1, seq_len, hidden_dim) * 0.02)

    def forward(self, x):
        h = self.net(x)                              # (B, 768)
        h = h.unsqueeze(1)                            # (B, 1, 768)
        h = h.expand(-1, self.pos_embeds.shape[1], -1)  # (B, 77, 768)
        return h + self.pos_embeds

encoder = ConditionEncoder().to(device)
n_total = sum(p.numel() for p in encoder.parameters())
n_train = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
model_mb = n_total * 4 / 1024 / 1024
print(f"ConditionEncoder: {n_total:,} total, {n_train:,} trainable params")
print(f"Model size (fp32): {model_mb:.2f} MB")

optimizer = AdamW(encoder.parameters(), lr=1e-4)

ConditionEncoder: 591,360 total, 591,360 trainable params
Model size (fp32): 2.26 MB


In [16]:
NUM_EPOCHS = 100
train_losses = []
best_loss = float("inf")

start_epoch = 0
checkpoint_path = Path("condition_encoder_checkpoint.pt")
if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location=device)
    encoder.load_state_dict(ckpt["encoder_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_loss = ckpt.get("best_loss", float("inf"))
    print(f"Resumed from epoch {start_epoch} (loss={ckpt['loss']:.6f})")

print(f"Starting training for {NUM_EPOCHS} epochs...")
t0 = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    encoder.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:3d}/{NUM_EPOCHS}", leave=False)

    for images, conditions, _ in pbar:
        images = images.to(device, non_blocking=True)
        conditions = conditions.to(device, non_blocking=True)

        with torch.no_grad():
            latents = vae.encode(images.half()).latent_dist.sample()
            latents = latents * vae.config.scaling_factor
            noise = torch.randn_like(latents)
            bsz = images.shape[0]
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (bsz,), device=device
            ).long()
            noisy = noise_scheduler.add_noise(latents, noise, timesteps)

        encoder_hidden = encoder(conditions)              # fp32
        pred = unet(noisy, timesteps, encoder_hidden.half(), return_dict=False)[0]  # fp16
        loss = F.mse_loss(pred.float(), noise.float())    # fp32

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * bsz
        pbar.set_postfix(loss=f"{loss.item():.6f}")

    avg_loss = epoch_loss / len(train_ds)
    train_losses.append(avg_loss)

    elapsed = time.time() - t0
    if epoch == 0:
        est_total = elapsed * NUM_EPOCHS
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | loss: {avg_loss:.6f} | "
              f"elapsed: {elapsed:.0f}s | est total: {est_total/60:.1f} min")
    else:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | loss: {avg_loss:.6f} | "
              f"elapsed: {elapsed:.0f}s")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(encoder.state_dict(), "condition_encoder_best.pt")

    if (epoch + 1) % 25 == 0 or epoch == NUM_EPOCHS - 1:
        torch.save({
            "epoch": epoch,
            "encoder_state_dict": encoder.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss,
            "best_loss": best_loss,
        }, "condition_encoder_checkpoint.pt")

# Save final
torch.save(encoder.state_dict(), "condition_encoder_final.pt")
total_time = time.time() - t0
print(f"\nTraining complete! {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Max VRAM used: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")

plt.figure(figsize=(10, 4))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()

Starting training for 100 epochs...


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_58/1639598169.py", line 19, in __getitem__
    img = Image.open(path).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
          ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/image_generation/train/images/8795.jpg'


In [19]:
encoder.eval()
dummy = torch.randn(1, 25, device=device)

with torch.no_grad():
    out = encoder(dummy)
print(f"ONNX test forward: input={dummy.shape} \u2192 output={out.shape}")

torch.onnx.export(
    encoder,
    (dummy,),
    "condition_encoder.onnx",
    input_names=["condition"],
    output_names=["encoder_hidden_states"],
    dynamic_axes={"condition": {0: "batch"},
                 "encoder_hidden_states": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

for fname in ["condition_encoder_best.pt", "condition_encoder_final.pt", "condition_encoder.onnx"]:
    if os.path.exists(fname):
        sz = os.path.getsize(fname) / 1024 / 1024
        print(f"{fname}: {sz:.2f} MB")

if os.path.exists("backend/app/models"):
    shutil.copy("condition_encoder_best.pt", "backend/app/models/condition_encoder.pt")
    shutil.copy("condition_encoder.onnx", "backend/app/models/condition_encoder.onnx")
    print("\u2713 Copied to backend/app/models/")
else:
    print("= backend/app/models/ not found — copy manually after download from Kaggle")

ONNX test forward: input=torch.Size([1, 25]) → output=torch.Size([1, 77, 768])


/tmp/ipykernel_160/497644416.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


condition_encoder.onnx: 2.26 MB
= backend/app/models/ not found — copy manually after download from Kaggle


In [ ]:
encoder.eval()
infer_scheduler.set_timesteps(20)
GEN_DIR = Path("outputs/condition_encoder/test")
GEN_DIR.mkdir(exist_ok=True, parents=True)

n_test = min(100, len(test_ds))
results = []

for i in tqdm(range(n_test), desc="Generating test images"):
    img_t, cond, sid = test_ds[i]
    target_rgb = test_valid[i]["color_rgb"]
    cond = cond.unsqueeze(0).to(device)

    with torch.no_grad():
        enc_hidden = encoder(cond).half()          # MLP fp32 → fp16 for UNet
        latent = torch.randn(1, 4, 64, 64, dtype=torch.float16, device=device)

        for t in infer_scheduler.timesteps:
            inp = infer_scheduler.scale_model_input(latent, t)
            pred = unet(inp, t.unsqueeze(0), enc_hidden, return_dict=False)[0]
            latent = infer_scheduler.step(pred, t, latent).prev_sample

        decoded = vae.decode(latent / vae.config.scaling_factor).sample

    gen = decoded[0].permute(1, 2, 0).cpu().float()  # CHW \u2192 HWC, fp16 \u2192 fp32
    gen_np = ((gen * 0.5 + 0.5).clamp(0, 1).numpy() * 255).astype(np.uint8)
    Image.fromarray(gen_np).save(GEN_DIR / f"{sid}_gen.png")

    real = ((img_t.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1).numpy() * 255).astype(np.uint8)
    Image.fromarray(real).save(GEN_DIR / f"{sid}_real.png")

    results.append({
        "id": sid,
        "target_rgb": target_rgb,
        "surface": test_valid[i]["surface"],
        "transparency": test_valid[i]["transparency"],
        "color_family": test_valid[i]["color_family"],
    })

with open(GEN_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Generated {n_test} pairs in {GEN_DIR}/")

In [ ]:
def color_distance(gen_np, target_rgb):
    mean = gen_np.mean(axis=(0, 1))
    t = np.array([target_rgb["r"], target_rgb["g"], target_rgb["b"]], dtype=np.float32)
    return float(np.sqrt(((mean - t) ** 2).sum()))

dists = []
pass_count = 0

for res in tqdm(results, desc="Color distance"):
    im = np.array(Image.open(GEN_DIR / f"{res['id']}_gen.png"))
    d = color_distance(im, res["target_rgb"])
    dists.append(d)
    if d < 100:
        pass_count += 1

dists = np.array(dists)
print(f"Color Distance:  mean={dists.mean():.2f}  median={np.median(dists):.2f}  std={dists.std():.2f}")
print(f"Pass Rate:       {pass_count}/{len(dists)} = {pass_count/len(dists)*100:.1f}%")
print(f"Paper GAN:       72.31 / 75.9%")
print(f"Paper cVAE:      134.49 / 46.6%")

In [ ]:
lpips_fn = lpips.LPIPS(net="alex").to("cpu")
fid = FrechetInceptionDistance(feature=2048).to("cpu")

lpips_scores = []

for res in tqdm(results, desc="LPIPS + FID"):
    gen = Image.open(GEN_DIR / f"{res['id']}_gen.png").resize((299, 299), Image.BILINEAR)
    real = Image.open(GEN_DIR / f"{res['id']}_real.png").resize((299, 299), Image.BILINEAR)

    t_gen = torch.from_numpy(np.array(gen, dtype=np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)
    t_real = torch.from_numpy(np.array(real, dtype=np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        d = lpips_fn(t_gen * 2 - 1, t_real * 2 - 1)
    lpips_scores.append(d.item())

    fid.update((t_gen * 255).to(dtype=torch.uint8), real=False)
    fid.update((t_real * 255).to(dtype=torch.uint8), real=True)

mean_lpips = float(np.mean(lpips_scores))
fid_score = float(fid.compute().item())
print(f"LPIPS: {mean_lpips:.4f}")
print(f"FID:   {fid_score:.2f}")

In [ ]:
n_grid = min(8, len(results))
fig, axes = plt.subplots(2, n_grid, figsize=(4 * n_grid, 8))

for i in range(n_grid):
    res = results[i]
    real = Image.open(GEN_DIR / f"{res['id']}_real.png")
    gen = Image.open(GEN_DIR / f"{res['id']}_gen.png")

    axes[0, i].imshow(real)
    axes[0, i].set_title(f"Real\n{res['surface']} / {res['color_family']}", fontsize=8)
    axes[0, i].axis("off")

    axes[1, i].imshow(gen)
    axes[1, i].set_title(f"gen\ndist={dists[i]:.0f}", fontsize=8)
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig(GEN_DIR / "comparison_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Grid saved to {GEN_DIR / 'comparison_grid.png'}")

In [ ]:
print("=" * 62)
print("EVALUATION RESULTS — ConditionEncoder MLP (Option 1)")
print("=" * 62)
print(f"{"Samples":30s} {len(results)} test images @ 512\u00d7512")
print(f"{"Training":30s} {NUM_EPOCHS} epochs, {len(train_ds)} samples, batch={BATCH_SIZE}")
print()
print(f"{"Metric":30s} {"Ours":>10s} {"GAN":>10s} {"cVAE":>10s}")
print("-" * 62)
print(f"{"Color Distance (mean)":30s} {dists.mean():>10.2f} {"72.31":>10s} {"134.49":>10s}")
print(f"{"Color Distance (median)":30s} {float(np.median(dists)):>10.2f} {"-":>10s} {"-":>10s}")
print(f"{"Pass Rate (d < 100)":30s} {pass_count/len(dists)*100:>9.1f}% {"75.9%":>9s} {"46.6%":>9s}")
print(f"{"LPIPS":30s} {mean_lpips:>10.4f} {"-":>10s} {"-":>10s}")
print(f"{"FID":30s} {fid_score:>10.2f} {"-":>10s} {"-":>10s}")
print("-" * 62)

beats_gan = dists.mean() < 72.31
verdict = "GOOD" if beats_gan else "POOR"
print()
print(f"Verdict: {verdict}")
if beats_gan:
    print(f"  Color distance ({dists.mean():.2f}) < paper GAN baseline (72.31)")
    print("  Proceed to OpenVINO conversion + backend integration.")
else:
    print(f"  Color distance ({dists.mean():.2f}) >= paper GAN baseline (72.31)")
    print("  Consider IP-Adapter (Option 2) or LoRA (Option 3) instead.")
print("=" * 62)